# Yahoo Finance ('yfinance') Web Scrapping Notebook
This notebook will be the introduction to working with the yfinance api for webscrapping. We will predominantly be using this for finding ETF price and volume data but we will need to adjust it for divdends. The best thing I think we can do is divide into sectors, but also pull full index (SPY, QQQ), we will then need to find proxies for different maturity bonds (long and short) and the equivalent of a money market (1-3 month treasuries). It may be a good idea to pull currency data from this as well, possibly the DXY index of USD strength.

## Libraries

In [53]:
import numpy as np
import pandas as pd
import altair as alt  

import yfinance as yf

# Disable the max rows limit in Altair
alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

ALright, let's start with just the basic pulls. I am pretty sure we should be able to pull the data using a batch pull but if not, let's pull them indvidually and add them to dataframes via a horizontal merge on date. We will only be focusing on closing prices and volume traded in the day. We can decide any data tansformations we want to use in the future.

In [54]:
# Let's create a list of sectors
sector_etfs = ["SPY","QQQ","XLF", "XLK", "XLU", "XLV", "XLE", "XLI", "XLB", "XLP", "XLY"]
prices = yf.download(sector_etfs, period='max', auto_adjust=True)[['Close','Volume']]

[*********************100%***********************]  11 of 11 completed


I'm going to separate the data pull from the data analysis so I don't have to continue to query the API as rate limiting is a known issue.

In [55]:
prices.dropna(inplace=True)
prices.head()



Price           Close                                                       \
Ticker            QQQ        SPY       XLB       XLE        XLF        XLI   
Date                                                                         
1999-03-10  43.128643  80.514305  6.032155  6.018416  12.424147  15.783931   
1999-03-11  43.339790  81.410225  6.096786  6.103729  12.484314  15.803363   
1999-03-12  42.284019  80.631187  6.070932  6.080457  12.506863  15.851920   
1999-03-15  43.498173  81.780258  6.083855  6.033927  12.544478  15.813067   
1999-03-16  43.867695  81.468636  6.058008  6.002906  12.363988  15.706224   

Price                                                  ...   Volume           \
Ticker            XLK        XLP       XLU        XLV  ...      SPY      XLB   
Date                                                   ...                     
1999-03-10  13.154649  14.296951  5.565428  19.005716  ...  3950000  53600.0   
1999-03-11  13.131361  14.437933  5.608498  19.131996  ...  6583700  27800.0   
1999-03-12  13.014946  14.562321  5.670033  19.026760  ...  5286500  15800.0   
1999-03-15  13.224492  14.686716  5.685414  19.089909  ...  5394400  13400.0   
1999-03-16  13.451499  14.753063  5.663878  19.047815  ...  4547500  15000.0   

Price                                                                  \
Ticker            XLE       XLF     XLI        XLK       XLP      XLU   
Date                                                                    
1999-03-10  3467600.0   91217.0  2700.0   884000.0   32600.0  68000.0   
1999-03-11  1818000.0  210993.0  1800.0  2432400.0   55700.0  19200.0   
1999-03-12  1208600.0  183911.0  2200.0  1672000.0   48100.0  15600.0   
1999-03-15  1046000.0   82969.0  1900.0   864200.0  246400.0  31800.0   
1999-03-16   546600.0  188466.0  4800.0  1517600.0   63700.0  20200.0   

Price                          
Ticker           XLV      XLY  
Date                           
1999-03-10   10700.0  11200.0  
1999-03-11   23600.0  26200.0  
1999-03-12   24600.0  27400.0  
1999-03-15  144800.0  14200.0  
1999-03-16    9100.0  12400.0  

[5 rows x 22 columns]

In [60]:
prices.tail()

Price            Close                                               \
Ticker             QQQ         SPY        XLB        XLE        XLF   
Date                                                                  
2026-02-13  601.919983  681.750000  53.310001  54.349998  51.650002   
2026-02-17  601.299988  682.849976  52.700001  53.750000  52.200001   
2026-02-18  605.789978  686.289978  53.020000  54.779999  52.590000   
2026-02-19  603.469971  684.479980  52.830002  55.180000  52.150002   
2026-02-20  608.809998  689.429993  52.959999  54.880001  52.490002   

Price                                                                 ...  \
Ticker             XLI         XLK        XLP        XLU         XLV  ...   
Date                                                                  ...   
2026-02-13  174.169998  139.559998  89.510002  46.500000  157.669998  ...   
2026-02-17  175.080002  139.479996  88.199997  46.380001  157.369995  ...   
2026-02-18  175.039993  140.910004  88.040001  45.610001  157.669998  ...   
2026-02-19  176.339996  140.210007  87.669998  46.110001  157.259995  ...   
2026-02-20  177.229996  140.880005  87.889999  46.330002  156.820007  ...   

Price         Volume                                                  \
Ticker           SPY         XLB         XLE         XLF         XLI   
Date                                                                   
2026-02-13  96267500  19918900.0  48392100.0  56449300.0  13617900.0   
2026-02-17  81354700  20404700.0  50499600.0  54064700.0  13810800.0   
2026-02-18  73570300  11992700.0  59196100.0  45224000.0  10623800.0   
2026-02-19  58649400  18340200.0  62513500.0  47344400.0  13431100.0   
2026-02-20  99952100  17809600.0  50799800.0  52919800.0  13103400.0   

Price                                                                   
Ticker             XLK         XLP         XLU         XLV         XLY  
Date                                                                    
2026-02-13  26637100.0  25649700.0  42451800.0  16647900.0  10836200.0  
2026-02-17  25743800.0  29819300.0  32424000.0  14881100.0   9623000.0  
2026-02-18  13705800.0  20422200.0  32841500.0  11886300.0   9719500.0  
2026-02-19  13484000.0  18447200.0  24172500.0  11226900.0   9482400.0  
2026-02-20  14940400.0  17908600.0  22622500.0  13367400.0  16641400.0  

[5 rows x 22 columns]

Alright, That data pull should work pretty easily let's take a look at much historical data we have. I thinmk the biggest concern is the fact that if we drop NA's we only have sector data from 2018. This does not give us a lot of exposure do different market regimes. I image this is do to the relatively new explosion of ETFs, and I imagine fixed income ETF's may contribute more to this problem. We may want to consider other options as surrogates.

In [56]:
min_date = min(prices.index)
print(f"The earliest date in our data is {min_date}.")

The earliest date in our data is 1999-03-10 00:00:00.


Ryan Peet brought up a really good idea of using mutual funds as proxies as they have been investment vehicles for a much longer period of time. So let's see if we can build the same datapull for mutual funds, and see how far back that data goes.  
Broad Market (SPY proxy): Vanguard 500 Index (VFINX) — data back to 1976, the gold standard  
Tech (XLK): Fidelity Select Technology (FSPTX) — inception 1981  
Healthcare (XLV): Fidelity Select Health Care (FSPHX) — inception 1981  
Energy (XLE): Fidelity Select Energy (FSENX) — inception 1981  
Financials (XLF): Fidelity Select Financial Services (FIDSX) — inception 1981  
Utilities (XLU): Fidelity Select Utilities (FSUTX) — inception 1981  
*Note* Industrials is really hard because it wasn't really a sector until the late 90's early '00s. It may be best to just drop it as the only one that works is a very heavily weighted subsection of industrials  
Industrials (XLI): Fidelity Select Industrials (FCYIX) — inception 1997 (this one is shorter I actually could only get data to 2019)
Industrials2 (XLI): Fidelity Select Defense & Aerospace (FSDAX)  
Consumer Staples (XLP): Fidelity Select Consumer Staples (FDFAX) — inception 1985  
Consumer Discretionary (XLY): Fidelity Select Retailing (FSRPX) as an imperfect proxy  
Materials (XLB): Fidelity Select Materials (FSDPX) — inception 1986  
Bonds (short-term): Vanguard Short-Term Bond Index (VBISX) or use direct Treasury yields from FRED  
Bonds (long-term): Vanguard Long-Term Bond Index (VBLTX) or TLT equivalent via Barclays index data from FRED  
Money market: 3-month T-bill rate from FRED is cleaner than any fund proxy  


In [57]:
# Let's create a list of sectors
sector_mfs = ["VFINX","FSPTX","FSPHX","FSENX","FIDSX","FSUTX","FSDAX","FDFAX","FSRPX","FSDPX","VBISX","VBLTX"]
prices_mfs = yf.download(sector_mfs, period='max', auto_adjust=True)[['Close','Volume']]

[*********************100%***********************]  12 of 12 completed


In [58]:
prices_mfs.dropna(inplace=True)
prices_mfs.head()



Price          Close                                                    \
Ticker         FDFAX     FIDSX     FSDAX     FSDPX     FSENX     FSPHX   
Date                                                                     
1994-02-28  1.262137  0.242021  0.113286  2.808203  1.871891  0.059039   
1994-03-01  1.255722  0.238856  0.112635  2.791357  1.858464  0.058852   
1994-03-02  1.246500  0.236731  0.112162  2.792652  1.866296  0.058629   
1994-03-03  1.244496  0.234227  0.112339  2.790062  1.873009  0.058312   
1994-03-04  1.242892  0.234936  0.113226  2.799132  1.870772  0.058535   

Price                                               ... Volume              \
Ticker         FSPTX     FSRPX     FSUTX     VBISX  ...  FSDAX FSDPX FSENX   
Date                                                ...                      
1994-02-28  0.064374  0.072639  2.635778  3.161900  ...    0.0   0.0   0.0   
1994-03-01  0.064389  0.072523  2.644474  3.161900  ...    0.0   0.0   0.0   
1994-03-02  0.064389  0.072523  2.653171  3.161900  ...    0.0   0.0   0.0   
1994-03-03  0.065005  0.073193  2.651722  3.158741  ...    0.0   0.0   0.0   
1994-03-04  0.065467  0.073631  2.658969  3.152419  ...    0.0   0.0   0.0   

Price                                                 
Ticker     FSPHX FSPTX FSRPX FSUTX VBISX VBLTX VFINX  
Date                                                  
1994-02-28   0.0   0.0   0.0   0.0   0.0   0.0     0  
1994-03-01   0.0   0.0   0.0   0.0   0.0   0.0     0  
1994-03-02   0.0   0.0   0.0   0.0   0.0   0.0     0  
1994-03-03   0.0   0.0   0.0   0.0   0.0   0.0     0  
1994-03-04   0.0   0.0   0.0   0.0   0.0   0.0     0  

[5 rows x 24 columns]

In [61]:
prices_mfs.tail()

Price           Close                                                       \
Ticker          FDFAX     FIDSX     FSDAX      FSDPX      FSENX      FSPHX   
Date                                                                         
2019-11-05  45.790173  5.098827  8.160415  45.300755  30.047281  10.517229   
2019-11-06  46.254982  5.098827  8.208674  45.019344  29.322943  10.542085   
2019-11-07  46.165192  5.135944  8.274485  45.490547  29.851286  10.591791   
2019-11-08  46.170475  5.135944  8.300809  45.641068  29.868330  10.666350   
2019-11-11  45.959194  5.131304  8.362230  45.634525  29.689373  10.641497   

Price                                                ... Volume              \
Ticker         FSPTX     FSRPX      FSUTX     VBISX  ...  FSDAX FSDPX FSENX   
Date                                                 ...                      
2019-11-05  6.591186  4.817155  58.969624  9.106742  ...    0.0   0.0   0.0   
2019-11-06  6.580578  4.808543  58.975842  9.115356  ...    0.0   0.0   0.0   
2019-11-07  6.626546  4.791319  58.627781  9.089509  ...    0.0   0.0   0.0   
2019-11-08  6.661906  4.785578  58.553204  9.089509  ...    0.0   0.0   0.0   
2019-11-11  6.676050  4.771224  58.149223  9.089509  ...    0.0   0.0   0.0   

Price                                                 
Ticker     FSPHX FSPTX FSRPX FSUTX VBISX VBLTX VFINX  
Date                                                  
2019-11-05   0.0   0.0   0.0   0.0   0.0   0.0     0  
2019-11-06   0.0   0.0   0.0   0.0   0.0   0.0     0  
2019-11-07   0.0   0.0   0.0   0.0   0.0   0.0     0  
2019-11-08   0.0   0.0   0.0   0.0   0.0   0.0     0  
2019-11-11   0.0   0.0   0.0   0.0   0.0   0.0     0  

[5 rows x 24 columns]